# Z15 拍照与板上识别

本 Notebook、camera_capture.bit、camera_capture.hwh 放在同一目录。将 sudoku_board_review.zip 上传到该目录并解压，得到 sudoku_board_review 文件夹。依次运行全部代码格；初始化后只重复最后一格拍照识别。识别在 ARM 上运行，求解与 LCD 尚未连接。


## 加载摄像头 Overlay（每次启动内核运行一次）


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate, MMIO

bitfile = os.path.join(os.getcwd(), 'camera_capture.bit')
overlay = Overlay(bitfile, download=True)
print('Overlay加载成功：', overlay.is_loaded())
print('可用IP：', list(overlay.ip_dict.keys()))

## 分配帧缓冲（运行一次，避免反复分配）


In [ ]:
WIDTH = 640
HEIGHT = 480
BYTES_PER_PIXEL = 2

frame = allocate(shape=(HEIGHT, WIDTH), dtype=np.uint16)
info = overlay.ip_dict['vdma']
vdma = MMIO(info['phys_addr'], info['addr_range'])
print('缓冲区地址：', hex(frame.physical_address))

## 定义拍照函数


In [ ]:
def take_photo(save_name=None):
    frame[:] = 0
    frame.flush()

    vdma.write(0x30, 0x04)
    time.sleep(0.2)
    vdma.write(0x30, 0x03)
    vdma.write(0xAC, frame.physical_address)
    vdma.write(0xA8, WIDTH * BYTES_PER_PIXEL)
    vdma.write(0xA4, WIDTH * BYTES_PER_PIXEL)
    vdma.write(0xA0, HEIGHT)

    time.sleep(1)
    status = vdma.read(0x34)
    vdma.write(0x30, 0x00)
    time.sleep(0.2)
    frame.invalidate()

    raw = np.array(frame, copy=True)
    r = (((raw >> 11) & 0x1F) * 255 // 31).astype(np.uint8)
    g = (((raw >> 5) & 0x3F) * 255 // 63).astype(np.uint8)
    b = ((raw & 0x1F) * 255 // 31).astype(np.uint8)
    image = np.dstack((r, g, b))

    if save_name:
        from PIL import Image
        Image.fromarray(image).save(save_name)
        print('已保存：', save_name)

    plt.figure(figsize=(10, 7))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    print('VDMA状态：', hex(status))
    return image

## 拍照并识别

保持棋盘四角完整入镜。每次生成独立结果目录，0 表示空白。必须核对原图和数字：双路一致仍可能认错，双路不同也可能误报。


In [ ]:
import time
import uuid
from pathlib import Path
import importlib.util

program = Path("sudoku_board_review/recognize.py").resolve()
if not program.is_file():
    raise FileNotFoundError("请先在当前目录解压 sudoku_board_review.zip")
spec = importlib.util.spec_from_file_location("board_recognizer", program)
board_recognizer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(board_recognizer)
process_photo = board_recognizer.process_photo

run_dir = Path("sudoku_board_review/results") / ("live_" + uuid.uuid4().hex[:8])
run_dir.mkdir(parents=True, exist_ok=True)
photo_path = run_dir / "capture.png"

start = time.perf_counter()

# 现场拍照并保存，随后直接在板上处理、识别
photo = take_photo(str(photo_path))
live_result = process_photo(photo_path, run_dir / "recognition")

print("拍照到识别总耗时：%.2f 秒" % (time.perf_counter() - start))
print("本次结果目录：", run_dir)
print("需要复核：", live_result["requires_review"])
print("可疑格：", live_result["review_cells"])
print("无论是否提示，都请核对照片中的原题数字。")
